# MatchFormer Epipolar Fine-tuning — Google Colab

This notebook fine-tunes MatchFormer with epipolar supervision baked in.  
Checkpoints are saved to **Google Drive** so training can be resumed if Colab disconnects.

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Upload your ScanNet data to Google Drive (see Cell 3)
3. Run cells **top to bottom** — on resume, skip to Cell 6

## Cell 1: Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# All checkpoints will be saved here — survives Colab restarts
DRIVE_CKPT_DIR = '/content/drive/MyDrive/final_proj/matchformer_checkpoints_run2'

import os
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')

Mounted at /content/drive
Checkpoint dir: /content/drive/MyDrive/final_proj/matchformer_checkpoints_run2


## Cell 2: Clone Repo & Install Dependencies

In [2]:
import os

REPO_URL = 'https://github.com/sid-raj-uc/match-former.git'  # your repo
REPO_DIR = '/content/final_proj'

if not os.path.exists(REPO_DIR):
    print('Repo not cloned — cloning')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !cd {REPO_DIR} && git pull

%cd /content/final_proj/MatchFormer

Repo not cloned — cloning
Cloning into '/content/final_proj'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 196 (delta 0), reused 1 (delta 0), pack-reused 192 (from 1)
Receiving objects: 100% (196/196), 121.36 MiB | 14.98 MiB/s, done.
Resolving deltas: 100% (83/83), done.
/content/final_proj/MatchFormer


In [3]:
# Install dependencies
!pip install -q pytorch-lightning einops kornia timm loguru yacs
!pip install -q opencv-python-headless
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 156.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 97.4 MB/s eta 0:00:00
Dependencies installed.


## Cell 3: ScanNet Data

Upload your exported ScanNet scenes to Google Drive. Expected structure:
```
MyDrive/final_proj/scannet/scans/
  scene0000_00/exported/{color, depth, pose, intrinsic}
  scene0001_00/exported/{color, depth, pose, intrinsic}
  ...
```
The training script picks up **all scene subdirs automatically** — no config changes needed.

In [ ]:
# Copy ScanNet data from Drive to local SSD — much faster than reading from Drive during training
# Local /content/ is an SSD; Drive has high per-file latency that kills dataloader throughput
import shutil, os, time

LOCAL_DATA = '/content/scannet_data'
if os.path.exists(LOCAL_DATA):
    print(f'Local data already exists at {LOCAL_DATA} — skipping copy')
else:
    print(f'Copying {DATA_ROOT} → {LOCAL_DATA} (this takes ~5-10 min, done once per session)...')
    t0 = time.time()
    shutil.copytree(DATA_ROOT, LOCAL_DATA)
    print(f'Done in {(time.time()-t0)/60:.1f} min')

# Point training at local copy
DATA_LOCAL = LOCAL_DATA
print(f'Training will read from: {DATA_LOCAL}')

In [5]:
# Point to the root scans directory — all scene subdirs will be used automatically
# Expected layout on Drive:
#   MyDrive/final_proj/scannet/scans/
#     scene0000_00/exported/{color, depth, pose, intrinsic}
#     scene0001_00/exported/{color, depth, pose, intrinsic}
#     ...
DATA_ROOT = '/content/drive/MyDrive/final_proj/data/scans'

import os, glob
scenes = sorted([d for d in os.listdir(DATA_ROOT)
                 if os.path.isdir(os.path.join(DATA_ROOT, d, 'exported', 'color'))])
print(f'Found {len(scenes)} scenes:')
total_frames = 0
for s in scenes:
    n = len(glob.glob(os.path.join(DATA_ROOT, s, 'exported', 'color', '*.jpg')))
    total_frames += n
    print(f'  {s}: {n} frames')
print(f'\nTotal frames: {total_frames}')

Found 11 scenes:
  scene0000_00: 5578 frames
  scene0001_00: 1544 frames
  scene0002_00: 5193 frames
  scene0003_00: 1736 frames
  scene0004_00: 929 frames
  scene0005_00: 1159 frames
  scene0006_00: 2160 frames
  scene0007_00: 1041 frames
  scene0008_00: 1038 frames
  scene0009_00: 980 frames
  scene0010_00: 2513 frames

Total frames: 23871


## Cell 4: Download Pretrained Weights

In [6]:
# Check if pretrained weight already exists (from a previous run or Drive)
WEIGHTS_PATH = 'model/weights/indoor-lite-LA.ckpt'
DRIVE_WEIGHTS = '/content/drive/MyDrive/final_proj/MatchFormer/model/weights/indoor-lite-LA.ckpt'

os.makedirs('model/weights', exist_ok=True)

if os.path.exists(DRIVE_WEIGHTS) and not os.path.exists(WEIGHTS_PATH):
    import shutil
    shutil.copy(DRIVE_WEIGHTS, WEIGHTS_PATH)
    print('Copied weights from Drive')
elif os.path.exists(WEIGHTS_PATH):
    print('Weights already present')
else:
    print('⚠️  Please upload indoor-lite-LA.ckpt to Drive at:', DRIVE_WEIGHTS)
    print('   or place it at:', WEIGHTS_PATH)

Copied weights from Drive


## Cell 5: Verify GPU & Run Sanity Check
Run this once to confirm the pipeline works before doing the full training.

In [7]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU!)"}')
print(f'CUDA available: {torch.cuda.is_available()}')

GPU: NVIDIA L4
CUDA available: True


In [8]:
# Sanity check: 5 pairs, 50 steps — should finish in ~1 min on L4
!python train_finetune.py \
    --overfit \
    --data_dir {DATA_ROOT} \
    --checkpoint_dir {DRIVE_CKPT_DIR}/overfit \
    --batch 4 \
    --steps 50

print('Sanity check complete — loss should be decreasing!')

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[Auto-resume] No checkpoint found — starting from scratch.
  Scenes found: 11
    scene0000_00: 5558 pairs
  Total pairs: 5
Dataset: 5 pairs
2026-03-16 20:56:34.614 | INFO     | model.lightning_loftr:__init__:51 - Load 'model/weights/indoor-lite-LA.ckpt' as pretrained checkpoint
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
/usr/local/lib/python3.12/di

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# L4 GPU: 24GB VRAM — batch 4, tau 50, bf16 mixed precision (~2x faster than fp32)
# Data is read from local SSD (DATA_LOCAL), checkpoints saved to Drive (DRIVE_CKPT_DIR)
!python train_finetune.py \
    --data_dir {DATA_LOCAL} \
    --checkpoint_dir {DRIVE_CKPT_DIR} \
    --steps 10000 \
    --tau 50.0 \
    --batch 4 \
    --workers 4 \
    --lr 1e-4 \
    --save_every 500 \
    --precision bf16

# To resume from last checkpoint (Colab reconnect):
# !python train_finetune.py \
#     --data_dir {DATA_LOCAL} \
#     --checkpoint_dir {DRIVE_CKPT_DIR} \
#     --steps 10000 \
#     --tau 50.0 --batch 4 --workers 4 --save_every 500 --precision bf16

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# L4 GPU: 24GB VRAM — batch 4, tau 50 (soft epipolar mask for robust multi-scene training)
# tau=50 is broader than tau=10 — prevents confidence collapse across diverse scenes
!python train_finetune.py \
    --data_dir {DATA_ROOT} \
    --checkpoint_dir {DRIVE_CKPT_DIR} \
    --steps 10000 \
    --tau 50.0 \
    --batch 4 \
    --workers 4 \
    --lr 1e-4 \
    --save_every 200

# To resume from last checkpoint (Colab reconnect):
# !python train_finetune.py \
#     --data_dir {DATA_ROOT} \
#     --checkpoint_dir {DRIVE_CKPT_DIR} \
#     --steps 10000 \
#     --tau 50.0 --batch 4 --workers 4 --save_every 200

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[Auto-resume] No checkpoint found — starting from scratch.
  Scenes found: 11
    scene0000_00: 5558 pairs
    scene0001_00: 1524 pairs
    scene0002_00: 5173 pairs
    scene0003_00: 1716 pairs
    scene0004_00: 909 pairs
    scene0005_00: 1139 pairs
    scene0006_00: 2140 pairs
    scene0007_00: 1021 pairs
    scene0008_00: 1018 pairs
    scene0009_00: 960 pairs
    scene0010_00: 2493 pairs
  Total pairs: 23651
Dataset: 23651 pairs
2026-03-16 20:59:45.359 | INFO     | model.lightning_loftr:__init__:51 - Load 'model/weights/indoor-lite-LA.ckpt' as pretrained checkpoint
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try instal

## Cell 7: Run Benchmark After Training

In [13]:
# After training finishes, copy the best checkpoint locally and run benchmark
import shutil

TRAINED_CKPT = f'{DRIVE_CKPT_DIR}/last.ckpt'
shutil.copy(TRAINED_CKPT, 'model/weights/epipolar-finetuned.ckpt')
print('Checkpoint copied.')

# Run benchmark with finetuned model
!python run_benchmark.py \
    --num_pairs 100 \
    --ckpt model/weights/epipolar-finetuned.ckpt \
    2>&1 | tee benchmark_finetuned_log.txt

# Copy results back to Drive
shutil.copy('benchmark_finetuned_log.txt', f'{DRIVE_CKPT_DIR}/benchmark_finetuned_log.txt')
print('Benchmark results saved to Drive!')

Checkpoint copied.
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
usage: run_benchmark.py [-h] [--num_pairs NUM_PAIRS]
run_benchmark.py: error: unrecognized arguments: --ckpt model/weights/epipolar-finetuned.ckpt
Benchmark results saved to Drive!
